In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ast import literal_eval
from collections import defaultdict
from itertools import combinations, product

In [2]:
basins_df = pd.read_csv("Basin60M.csv")

# Direct filtering on Basins.csv instead of using filtered_gmaca_candidates.csv
basins_filtered = basins_df[(basins_df['NumFixedPointAttractors'] >= 13) & 
                           (basins_df['NumFixedPointAttractors'] <= 16)].copy()
basins_filtered.reset_index(drop=True, inplace=True)
basins_filtered.sort_values('NumFixedPointAttractors', ascending=False, inplace=True)
basins_filtered.reset_index(drop=True, inplace=True)

print(f"Filtered to {len(basins_filtered)} GMACAs with 13–16 fixed-point attractors.")
basins_filtered[['RuleVectorID', 'NumFixedPointAttractors']].head(10)

Filtered to 25536 GMACAs with 13–16 fixed-point attractors.


C:\Users\jyoti\AppData\Local\Temp\ipykernel_3520\3843681890.py:1: DtypeWarning: Columns (29,30) have mixed types. Specify dtype option on import or set low_memory=False.
  basins_df = pd.read_csv("Basin60M.csv")


,RuleVectorID,NumFixedPointAttractors
0,RV_238-140-78-252-202-206-12-207,16
1,RV_192-197-92-252-196-200-78-12,16
2,RV_192-212-206-72-141-221-12-192,16
3,RV_172-237-198-216-78-221-200-136,16
4,RV_197-252-77-221-200-220-216-236,16
5,RV_221-156-207-238-12-132-236-221,16
6,RV_192-202-236-142-228-68-132-232,16
7,RV_196-228-141-232-172-198-200-136,16
8,RV_192-196-237-201-142-236-77-92,16
9,RV_136-207-68-140-192-68-200-136,16


In [ ]:
basins_filtered.to_csv("MACA_Basin_Config.csv", index=False)
print("Filtered basins exported to MACA_Basin_Config.csv")

In [ ]:
import numpy as np
import random
from itertools import combinations

def hamming_distance(a, b):
    return sum(c1 != c2 for c1, c2 in zip(a, b))

def parse_basin_data(row):
    basins = {}
    basin_cols = [col for col in row.index if col.startswith('Basin') and row[col] not in ['', '-1', 'TRUNCATED']]
    
    for col in basin_cols:
        basin_data = str(row[col])
        if basin_data == '' or basin_data == '-1' or basin_data == 'TRUNCATED':
            continue
            
        if '|' in basin_data:
            basin_part, attractor_part = basin_data.split('|', 1)
            basin_states = basin_part.split('-') if basin_part else []
            attractor_states = attractor_part.split('-')
        else:
            basin_states = []
            attractor_states = basin_data.split('-')
        
        attractor_key = attractor_states[0]
        
        # Convert basin states to binary
        basin_binary = []
        for state_num in basin_states:
            if state_num.strip():
                binary_state = format(int(state_num), '08b')
                basin_binary.append(binary_state)
        
        # Convert attractor states to binary
        attractor_binary = []
        for state_num in attractor_states:
            if state_num.strip():
                binary_state = format(int(state_num), '08b')
                attractor_binary.append(binary_state)
        
        # Combine basin states and attractor states
        all_states = basin_binary + attractor_binary
        
        if all_states:
            basins[attractor_key] = all_states
    
    return basins

def analyze_gmaca_row(row, min_basin_size=2):
    rule_vector_id = row['RuleVectorID']
    basins = parse_basin_data(row)
    valid_basins = {k: v for k, v in basins.items() if len(v) >= min_basin_size}
    
    if not valid_basins:
        return None
    
    result = {
        'RuleVectorID': rule_vector_id,
        **{f'B{i}': '-1' for i in range(1, 17)},
        **{f'B{i}_Attractor': '-1' for i in range(1, 17)},
        **{f'H{i}': -1 for i in range(1, 17)},
        **{f'H{i}_Attractor': '-1' for i in range(1, 17)},
        'Avg_PE_Bit_No': 0,
        'Max_Common_PE_Bit': 0,
        'Min_Hamming': -1,
        'Max_Hamming': -1
    }
    
    all_pe_counts = []
    all_basin_pe_data = []
    weighted_pe_sum = 0
    total_states = 0
    
    sorted_basins = sorted(valid_basins.items(), key=lambda x: len(x[1]), reverse=True)
    
    for basin_idx, (basin_key, states) in enumerate(sorted_basins):
        if basin_idx >= 16:
            break
            
        basin_size = len(states)
        state_matrix = np.array([[int(bit) for bit in state] for state in states])
        
        # Store the attractor binary representation (basin_key converted to 8-bit binary)
        attractor_binary = format(int(basin_key), '08b')
        
        pe_positions = []
        pe_bit_map = {}
        
        for bit_pos in range(8):
            bit_column = state_matrix[:, bit_pos]
            if len(np.unique(bit_column)) == 1:
                pe_positions.append(bit_pos)
                pe_bit_map[bit_pos] = str(bit_column[0])
        
        if pe_positions:
            positions_str = ''.join(str(pos) for pos in pe_positions)
            values_list = []
            for pos in range(8):
                if pos in pe_bit_map:
                    values_list.append(pe_bit_map[pos])
                else:
                    values_list.append('-')
            values_str = ''.join(values_list)
            result[f'B{basin_idx + 1}'] = f"{positions_str}|{values_str}"
        else:
            result[f'B{basin_idx + 1}'] = '-1'
        
        # Store the attractor for this basin
        result[f'B{basin_idx + 1}_Attractor'] = attractor_binary
        
        pe_count = len(pe_positions)
        all_pe_counts.append(pe_count)
        
        all_basin_pe_data.append([str(pos) for pos in pe_positions])
        
        weighted_pe_sum += pe_count * basin_size
        total_states += basin_size
        
        if len(states) > 1:
            basin_hamming_distances = [hamming_distance(a, b) for a, b in combinations(states, 2)]
            avg_hamming_for_basin = np.mean(basin_hamming_distances)
            result[f'H{basin_idx + 1}'] = avg_hamming_for_basin
        else:
            result[f'H{basin_idx + 1}'] = 0
        
        # Store the attractor for this basin's Hamming distance as well
        result[f'H{basin_idx + 1}_Attractor'] = attractor_binary
    
    if len(sorted_basins) >= 2:
        inter_basin_distances = []
        basin_states = [states for _, states in sorted_basins[:16]]
        
        for i in range(len(basin_states)):
            for j in range(i + 1, len(basin_states)):
                basin_a_states = basin_states[i]
                basin_b_states = basin_states[j]
                
                # Sample 10 pairs between these two basins
                for _ in range(10):
                    state_a = random.choice(basin_a_states)
                    state_b = random.choice(basin_b_states)
                    distance = hamming_distance(state_a, state_b)
                    inter_basin_distances.append(distance)
        
        if inter_basin_distances:
            result['Min_Hamming'] = min(inter_basin_distances)
            result['Max_Hamming'] = max(inter_basin_distances)
    
    if all_pe_counts:
        result['Avg_PE_Bit_No'] = weighted_pe_sum / total_states if total_states > 0 else 0
    
    if all_basin_pe_data:
        common_pe_positions = set(all_basin_pe_data[0])
        for basin_pe in all_basin_pe_data[1:]:
            common_pe_positions = common_pe_positions.intersection(set(basin_pe))
        result['Max_Common_PE_Bit'] = len(common_pe_positions)
    else:
        result['Max_Common_PE_Bit'] = 0
    
    return result

In [4]:
# Configuration
MIN_BASIN_SIZE = 0

print(f"Starting analysis of {len(basins_filtered)} GMACAs...")

results = []
for index, row in basins_filtered.iterrows():
    result = analyze_gmaca_row(row, min_basin_size=MIN_BASIN_SIZE)
    results.append(result)

results_df = pd.DataFrame(results)

column_order = (['RuleVectorID'] + 
               [item for i in range(1, 17) for item in [f'B{i}', f'B{i}_Attractor']] +
               [item for i in range(1, 17) for item in [f'H{i}', f'H{i}_Attractor']] +
               ['Avg_PE_Bit_No', 'Max_Common_PE_Bit', 'Min_Hamming', 'Max_Hamming'])

results_df = results_df[column_order]

output_filename = "Hamming_Basin_Analysis_Advanced.csv"
results_df.to_csv(output_filename, index=False)

print(f"Results saved to: {output_filename}")
print(f"Processed: {len(results_df)} GMACAs, {len(results_df.columns)} columns")

# Summary Statistics
print(f"\nSummary Statistics:")
metric_cols = ['Avg_PE_Bit_No', 'Max_Common_PE_Bit', 'Min_Hamming', 'Max_Hamming']
for col in metric_cols:
    vals = results_df[col]
    if col in ['Min_Hamming', 'Max_Hamming']:
        valid_vals = vals[vals != -1]
        if len(valid_vals) > 0:
            print(f"  {col:15}: Mean={valid_vals.mean():.3f}, Range=[{valid_vals.min():.0f}, {valid_vals.max():.0f}]")
    else:
        print(f"  {col:15}: Mean={vals.mean():.3f}, Range=[{vals.min():.3f}, {vals.max():.3f}]")

# Hamming Distance Summary
h_cols = [f'H{i}' for i in range(1, 17)]
all_h_values = []
for col in h_cols:
    valid_vals = results_df[col][results_df[col] != -1]
    all_h_values.extend(valid_vals)

if all_h_values:
    all_h_values = np.array(all_h_values)
    print(f"\nPer-Basin Hamming Distances:")
    print(f"  Mean={all_h_values.mean():.3f}, Range=[{all_h_values.min():.0f}, {all_h_values.max():.0f}]")
    print(f"  Total measurements: {len(all_h_values)}")

# PE Bit Summary
b_cols = [f'B{i}' for i in range(1, 17)]
valid_b_count = sum(len(results_df[col][results_df[col] != '-1']) for col in b_cols)
print(f"\nPE Bit Analysis:")
print(f"  Total basins with PE bits: {valid_b_count}")
print(f"  Format: 'positions|8bit_values' (e.g., '025|1-0--1--')")

Starting analysis of 25536 GMACAs...
Results saved to: Hamming_Basin_Analysis_Advanced.csv
Processed: 25536 GMACAs, 69 columns

Summary Statistics:
  Avg_PE_Bit_No  : Mean=1.855, Range=[0.250, 3.438]
  Max_Common_PE_Bit: Mean=0.211, Range=[0.000, 3.000]
  Min_Hamming    : Mean=1.000, Range=[1, 1]
  Max_Hamming    : Mean=7.964, Range=[7, 8]

Per-Basin Hamming Distances:
  Mean=2.339, Range=[1, 5]
  Total measurements: 370232

PE Bit Analysis:
  Total basins with PE bits: 328048
  Format: 'positions|8bit_values' (e.g., '025|1-0--1--')
Results saved to: Hamming_Basin_Analysis_Advanced.csv
Processed: 25536 GMACAs, 69 columns

Summary Statistics:
  Avg_PE_Bit_No  : Mean=1.855, Range=[0.250, 3.438]
  Max_Common_PE_Bit: Mean=0.211, Range=[0.000, 3.000]
  Min_Hamming    : Mean=1.000, Range=[1, 1]
  Max_Hamming    : Mean=7.964, Range=[7, 8]

Per-Basin Hamming Distances:
  Mean=2.339, Range=[1, 5]
  Total measurements: 370232

PE Bit Analysis:
  Total basins with PE bits: 328048
  Format: 'posit